# 🧠 DSA Lab 15: Heaps, Priority Queues and Heap Sort
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 15 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 📖 Part A: What Is a Heap?

A **binary heap** is a binary tree with two properties:

1. **Shape property:** it is a **complete binary tree** (all levels full except possibly the last, which is filled from the left).
2. **Heap property:** in a **max-heap**, every parent ≥ its children, so the **largest** key is at the root. In a **min-heap**, every parent ≤ its children, so the **smallest** key is at the root.

```text
      Max-heap                       Min-heap
         90                             5
       /    \                         /   \
     70      80                      10     8
    /  \    /                       / \    /
  40   60  50                      20  15 30

array: [90, 70, 80, 40, 60, 50]   array: [5, 10, 8, 20, 15, 30]
```

> **⚠️ A heap is not sorted**
>
> Only parent–child order is guaranteed. Siblings and cousins can be in any order. 80 is in level 1 and 60 in level 2, but nothing says 70 > 80. This partial order is what makes heaps fast to maintain.

### Array Representation

Because a heap is complete, we store it in an array **without gaps** (as in Lab 13). For index i: **left = 2i + 1**, **right = 2i + 2**, **parent = (i − 1) // 2**. The height of a heap with n nodes is ⌊log₂ n⌋, which is why its operations are O(log n).

## ⚙️ Part B: Heap Operations (Min-Heap)

### Insert: Add at the End, then Sift Up

Put the new key at the end of the array (keeps the shape). Then, while it is smaller than its parent, **swap it with the parent** (“bubble up”).

**Sift-up**

```text
Insert 3 into [5, 10, 8, 20, 15, 30]
[5, 10, 8, 20, 15, 30, 3]      3 at index 6, parent index 2 (8)  -> swap
[5, 10, 3, 20, 15, 30, 8]      3 at index 2, parent index 0 (5)  -> swap
[3, 10, 5, 20, 15, 30, 8]      3 is the new root
```

### Extract Min: Move the Last to the Root, then Sift Down

The minimum is at index 0. Replace the root with the **last element**, remove the last slot, and then, while the new root is larger than its **smaller child**, swap them (“bubble down”).

In [ ]:
class MinHeap:
    def __init__(self):
        self.a = []

    def __len__(self):
        return len(self.a)

    def peek(self):
        return self.a[0] if self.a else None              # O(1)

    def push(self, x):                                    # O(log n)
        self.a.append(x)
        self._sift_up(len(self.a) - 1)

    def pop(self):                                        # O(log n)
        if not self.a:
            raise IndexError("pop from empty heap")
        top = self.a[0]
        last = self.a.pop()
        if self.a:
            self.a[0] = last
            self._sift_down(0)
        return top

    def _sift_up(self, i):
        while i > 0:
            parent = (i - 1) // 2
            if self.a[i] < self.a[parent]:
                self.a[i], self.a[parent] = self.a[parent], self.a[i]
                i = parent
            else:
                break

    def _sift_down(self, i):
        n = len(self.a)
        while True:
            smallest, l, r = i, 2 * i + 1, 2 * i + 2
            if l < n and self.a[l] < self.a[smallest]:
                smallest = l
            if r < n and self.a[r] < self.a[smallest]:
                smallest = r
            if smallest == i:
                break
            self.a[i], self.a[smallest] = self.a[smallest], self.a[i]
            i = smallest

h = MinHeap()
for x in [20, 5, 15, 30, 8, 10]:
    h.push(x)
    print(f"push {x:>2}: {h.a}")
print("pop order:", [h.pop() for _ in range(len(h))])

| Operation | Time | Why |
|---|---|---|
| peek | O(1) | Root of the heap |
| push | O(log n) | Sift-up along one path |
| pop | O(log n) | Sift-down along one path |
| build (heapify) | O(n) | Most nodes are near the bottom |
| search for arbitrary key | O(n) | Heaps are not ordered for searching |

## 🏗️ Part C: Building a Heap in O(n)

Inserting n items one by one costs O(n log n). **Bottom-up heapify** is faster: leaves are already heaps, so call sift-down on every non-leaf node from index n//2 − 1 down to 0. Most nodes are near the bottom and sift down only a little, giving a total of **O(n)**.

In [ ]:
def heapify(arr):                     # turn arr into a MIN-heap in place
    h = MinHeap()
    h.a = arr
    for i in range(len(arr) // 2 - 1, -1, -1):
        h._sift_down(i)
    return arr

data = [9, 4, 7, 1, 8, 2, 6, 3]
print(heapify(data))

## 📶 Part D: Heap Sort

Heap sort uses a **max-heap**: build it in O(n), then repeatedly **swap the root (maximum) with the last element of the heap**, shrink the heap by one, and sift down. The array becomes sorted in ascending order **in place**.

In [ ]:
def sift_down_max(a, i, n):
    while True:
        largest, l, r = i, 2 * i + 1, 2 * i + 2
        if l < n and a[l] > a[largest]:
            largest = l
        if r < n and a[r] > a[largest]:
            largest = r
        if largest == i:
            return
        a[i], a[largest] = a[largest], a[i]
        i = largest

def heap_sort(a):
    n = len(a)
    for i in range(n // 2 - 1, -1, -1):        # 1. build a max-heap: O(n)
        sift_down_max(a, i, n)
    for end in range(n - 1, 0, -1):            # 2. move max to the end: n x O(log n)
        a[0], a[end] = a[end], a[0]
        sift_down_max(a, 0, end)
    return a

print(heap_sort([12, 3, 17, 8, 34, 25, 1]))

| Property | Heap sort |
|---|---|
| Time (all cases) | O(n log n) |
| Extra space | O(1) |
| Stable? | No |
| Used in | Introsort’s fallback (C++ `std::sort`), embedded systems |

## 🐍 Part E: Python’s `heapq` Module

`heapq` provides a **min-heap** on a plain list. For a **max-heap**, push negated values or tuples with negated priority.

In [ ]:
import heapq

marks = [67, 89, 45, 92, 78, 55, 81]
heap = marks[:]
heapq.heapify(heap)                         # O(n)
heapq.heappush(heap, 50)
print("smallest:", heap[0], "| popped:", heapq.heappop(heap))
print("top 3:", heapq.nlargest(3, marks), "| bottom 2:", heapq.nsmallest(2, marks))

max_heap = []
for m in marks:
    heapq.heappush(max_heap, -m)            # negate for max-heap
print("max:", -heapq.heappop(max_heap))

tasks = []
heapq.heappush(tasks, (2, "Prepare viva"))
heapq.heappush(tasks, (1, "Submit DSA lab"))
heapq.heappush(tasks, (3, "Watch PSL match"))
while tasks:
    print(heapq.heappop(tasks))

## 🧩 Part F: Classic Heap Patterns

### Top-K from a Stream: Keep a Min-Heap of Size K

To keep the k largest values seen so far, keep a **min-heap of size k**. Its root is the smallest of the top k; a new value enters only if it beats that root. Time O(n log k), memory O(k): perfect for huge streams.

In [ ]:
def top_k(stream, k):
    heap = []
    for x in stream:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)       # pop smallest, push x
    return sorted(heap, reverse=True)

import random
scores = (random.randint(0, 10_000) for _ in range(100_000))   # a generator: a "stream"
print(top_k(scores, 5))

### Running Median with Two Heaps

Keep the smaller half of the numbers in a **max-heap** and the larger half in a **min-heap**, balanced in size. The median is at the top of one or both heaps. Each new number costs O(log n).

In [ ]:
class RunningMedian:
    def __init__(self):
        self.low = []                         # max-heap (store negatives)
        self.high = []                        # min-heap

    def add(self, x):
        heapq.heappush(self.low, -x)
        heapq.heappush(self.high, -heapq.heappop(self.low))   # move the largest of low
        if len(self.high) > len(self.low):
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def median(self):
        if len(self.low) > len(self.high):
            return -self.low[0]
        return (-self.low[0] + self.high[0]) / 2

rm = RunningMedian()
for bpm in [72, 90, 65, 80, 100, 70]:
    rm.add(bpm)
    print(f"added {bpm:>3} -> median {rm.median()}")

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <queue>
#include <vector>
#include <algorithm>
using namespace std;

int main() {
    priority_queue<int> maxHeap;                              // MAX-heap by default
    priority_queue<int, vector<int>, greater<int>> minHeap;   // MIN-heap
    for (int x : {20, 5, 15, 30, 8}) { maxHeap.push(x); minHeap.push(x); }
    cout << maxHeap.top() << " " << minHeap.top() << endl;   // 30 5

    vector<int> v = {12, 3, 17, 8, 34, 25, 1};
    make_heap(v.begin(), v.end());          // O(n) build, max-heap
    sort_heap(v.begin(), v.end());          // heap sort
    for (int x : v) cout << x << " ";
    cout << endl;

    // Patients: (severity, name); highest severity first
    priority_queue<pair<int, string>> er;
    er.push({2, "Ali"}); er.push({5, "Sara"}); er.push({3, "Usman"});
    while (!er.empty()) { cout << er.top().second << " "; er.pop(); }
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Is It a Heap?
Write `is_min_heap(arr)` that checks whether an array satisfies the min-heap property (every parent ≤ its children). It must be O(n).

```text
[1, 3, 2, 7, 4] → True,  [1, 3, 2, 0] → False
```

In [ ]:
def is_min_heap(arr):
    # TODO: check every parent against its children
    pass

In [ ]:
# Test cell: run after your solution
import heapq, random
assert is_min_heap([1, 3, 2, 7, 4]) is True and is_min_heap([1, 3, 2, 0]) is False
assert is_min_heap([]) is True and is_min_heap([5]) is True
for _ in range(20):
    r = [random.randint(0, 99) for _ in range(30)]; heapq.heapify(r)
    assert is_min_heap(r)
print("✅ Task 1 passed")

### Task 2 🟢 Last Stone Weight
You have stones with given weights. Each turn, take the **two heaviest** stones and smash them: if equal, both disappear; otherwise the lighter is destroyed and the heavier loses that weight. Return the weight of the last stone (0 if none). Use `heapq` as a max-heap.

```text
[2, 7, 4, 1, 8, 1] → 1
```

In [ ]:
import heapq

def last_stone(weights):
    heap = [-w for w in weights]
    heapq.heapify(heap)
    # TODO
    return -heap[0] if heap else 0

In [ ]:
# Test cell: run after your solution
assert last_stone([2, 7, 4, 1, 8, 1]) == 1
assert last_stone([3, 3]) == 0 and last_stone([5]) == 5 and last_stone([]) == 0
print("✅ Task 2 passed")

### Task 3 🟡 Max-Heap Class with Update
Write `MaxHeap` (your own, not `heapq`) with `push`, `pop`, `peek` and `__len__`. Then add `increase_key(i, new_value)`, which raises the value at index i and restores the heap (sift up).

Priority updates are needed in schedulers and in Dijkstra’s algorithm.

In [ ]:
class MaxHeap:
    def __init__(self):
        self.a = []

    def __len__(self):
        return len(self.a)

    def peek(self):
        return self.a[0] if self.a else None

    def push(self, x):
        # TODO
        pass

    def pop(self):
        # TODO
        pass

    def increase_key(self, i, new_value):
        # TODO
        pass

In [ ]:
# Test cell: run after your solution
import random
h = MaxHeap()
data = [random.randint(0, 999) for _ in range(200)]
for x in data:
    h.push(x)
assert h.peek() == max(data)
h_copy = h.a[:-1]
h.increase_key(len(h) - 1, 5000); assert h.peek() == 5000
out = [h.pop() for _ in range(len(h))]
expected = sorted(h_copy + [5000], reverse=True)
assert out == expected
print("✅ Task 3 passed")

### Task 4 🟡 Top-K Frequent Hashtags
Given a list of hashtags, return the `k` most frequent ones (ties broken alphabetically). Count with a dictionary, then use a heap (`heapq.nsmallest` with key `(-count, tag)` or your own size-k heap). Do not sort the whole dictionary.

```text
["#PSL", "#DSA", "#PSL", "#Okara", "#DSA", "#PSL"], k = 2 → ["#PSL", "#DSA"]
```

In [ ]:
import heapq

def top_k_tags(tags, k):
    count = {}
    # TODO: count, then pick k using a heap
    pass

In [ ]:
# Test cell: run after your solution
assert top_k_tags(["#PSL", "#DSA", "#PSL", "#Okara", "#DSA", "#PSL"], 2) == ["#PSL", "#DSA"]
assert top_k_tags(["#b", "#a"], 2) == ["#a", "#b"]
assert top_k_tags([], 3) == []
print("✅ Task 4 passed")

### Task 5 🟡 Merge K Sorted Result Lists
Each of k sections has a merit list sorted in ascending order. Merge them into one sorted list in **O(N log k)** (N = total items) using a min-heap holding the current front of each list as `(value, list_index, element_index)`.

This is exactly how databases merge sorted runs during external sorting.

```text
[[1, 4, 7], [2, 5, 8], [0, 9]] → [0, 1, 2, 4, 5, 7, 8, 9]
```

In [ ]:
import heapq

def merge_k(lists):
    heap = [(lst[0], i, 0) for i, lst in enumerate(lists) if lst]
    heapq.heapify(heap)
    out = []
    # TODO: pop the smallest, push the next element from the same list
    return out

In [ ]:
# Test cell: run after your solution
import random
assert merge_k([[1, 4, 7], [2, 5, 8], [0, 9]]) == [0, 1, 2, 4, 5, 7, 8, 9]
assert merge_k([]) == [] and merge_k([[], [3]]) == [3]
ls = [sorted(random.sample(range(1000), 50)) for _ in range(8)]
assert merge_k(ls) == sorted(x for l in ls for x in l)
print("✅ Task 5 passed")

### Task 6 🔴 K Closest Petrol Pumps
Petrol pumps are at coordinates `(x, y)` relative to your car at (0, 0). Return the `k` closest pumps (by Euclidean distance; ties by coordinates), using a **max-heap of size k** so that the solution is O(n log k).

Hint: compare squared distances (no square roots needed); push `(-dist², -x, -y)` into `heapq`.

```text
[(1, 3), (-2, 2), (5, 8), (0, 1)], k = 2 → [(0, 1), (-2, 2)]
```

In [ ]:
import heapq

def k_closest(points, k):
    heap = []
    # TODO: max-heap of size k using negated tuples
    return sorted(((-nx, -ny) for _, nx, ny in heap), key=lambda p: (p[0] ** 2 + p[1] ** 2, p))

In [ ]:
# Test cell: run after your solution
import random
assert k_closest([(1, 3), (-2, 2), (5, 8), (0, 1)], 2) == [(0, 1), (-2, 2)]
assert k_closest([(3, 3)], 1) == [(3, 3)] and k_closest([], 2) == []
pts = [(random.randint(-50, 50), random.randint(-50, 50)) for _ in range(300)]
exp = sorted(pts, key=lambda p: (p[0] ** 2 + p[1] ** 2, p))[:10]
assert [p[0] ** 2 + p[1] ** 2 for p in k_closest(pts, 10)] == [p[0] ** 2 + p[1] ** 2 for p in exp]
print("✅ Task 6 passed")

### Task 7 🔴 CPU Task Scheduler Simulation
Tasks arrive as `(arrival_time, duration, name)`. A single CPU always runs, among the tasks that have **already arrived**, the one with the **shortest duration** (ties: earlier arrival, then name). Tasks run to completion. Return the order of task names.

Sort tasks by arrival; keep a min-heap of available tasks keyed by `(duration, arrival, name)`; when the CPU is idle and nothing is available, jump time to the next arrival.

```text
[(0, 8, "A"), (1, 4, "B"), (2, 9, "C"), (3, 5, "D")] → ["A", "B", "D", "C"]
```

In [ ]:
import heapq

def schedule(tasks):
    tasks = sorted(tasks)
    heap, order = [], []
    time, i = 0, 0
    # TODO: add arrived tasks to the heap; run the shortest; handle idle CPU
    return order

In [ ]:
# Test cell: run after your solution
assert schedule([(0, 8, "A"), (1, 4, "B"), (2, 9, "C"), (3, 5, "D")]) == ["A", "B", "D", "C"]
assert schedule([(5, 2, "X"), (0, 1, "Y")]) == ["Y", "X"]
assert schedule([]) == []
assert schedule([(0, 3, "A"), (0, 3, "B"), (0, 1, "C")]) == ["C", "A", "B"]
print("✅ Task 7 passed")

## 🔬 Bonus: Huffman Coding (How ZIP Compresses Text)

In [ ]:
# Bonus: run after completing the tasks
import heapq
from collections import Counter

text = "data structures and algorithms at university of okara"
freq = Counter(text)
heap = [[f, i, {ch: ""}] for i, (ch, f) in enumerate(freq.items())]
heapq.heapify(heap)
counter = len(heap)
while len(heap) > 1:
    f1, _, c1 = heapq.heappop(heap)          # two least frequent groups
    f2, _, c2 = heapq.heappop(heap)
    merged = {ch: "0" + code for ch, code in c1.items()}
    merged.update({ch: "1" + code for ch, code in c2.items()})
    heapq.heappush(heap, [f1 + f2, counter, merged]); counter += 1
codes = heap[0][2]
bits = sum(len(codes[ch]) for ch in text)
print("codes:", dict(sorted(codes.items(), key=lambda kv: len(kv[1]))))
print(f"original: {len(text) * 8} bits, Huffman: {bits} bits ({bits / (len(text) * 8):.0%})")

---
## 🎉 Lab 15 Complete!

Next: **Lab 16: Graphs, BFS, DFS and Shortest Paths**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab15/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)